# Week 2: Feature Engineering & Predictive Modeling
## Customer Churn Prediction & Lifetime Value (LTV) Engine

### Objectives:
1. **Feature Engineering**: Construct business-driven behavioral, financial, and engagement features.
2. **Predictive Modeling**: Train multiple classification algorithms (Logistic Regression, Random Forest, XGBoost).
3. **Model Evaluation**: Benchmark Precision, Recall, F1-Score, and ROC-AUC for churn detection.
4. **Model Explainability (SHAP)**: Interpret global feature importance and local individual churn risk.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# Add path to import week_2 modules
sys.path.insert(0, os.path.abspath('..'))
from week_2.data_loader import load_raw_data
from week_2.feature_engineering import FeatureEngineer
from week_2.train_and_evaluate import ModelTrainer
from week_2.shap_explainability import ShapExplainer

%matplotlib inline

## 1. Data Ingestion & Overview

In [2]:
df_raw = load_raw_data()
print(f"Dataset Shape: {df_raw.shape}")
df_raw.head()

## 2. Feature Engineering
We engineer:
- **Tenure Cohorts** (`tenure_cohort`): Lifecycle buckets (0-12m, 12-24m, etc.)
- **Financial Dynamics**: `monthly_to_total_ratio`, `avg_historical_monthly`, and `bill_shock`
- **Service Bundling**: `service_bundle_count`, `has_security_bundle`, `has_streaming_bundle`
- **Contract & Payment Risks**: `is_month_to_month`, `is_electronic_check`, `is_automatic_payment`
- **Household Demographics**: `has_family`, `is_senior_alone`

In [3]:
fe = FeatureEngineer()
X_train, X_test, y_train, y_test, feature_names = fe.fit_transform(df_raw)

print(f"Total features engineered: {len(feature_names)}")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
print(f"Training set churn distribution:\n{y_train.value_counts(normalize=True)}")

## 3. Model Training & Evaluation Benchmark
We train and benchmark:
1. **Logistic Regression** (Linear baseline with balanced weighting)
2. **Random Forest Classifier** (Bagged decision tree ensemble)
3. **XGBoost Classifier** (Gradient boosted trees with positive class weighting)

In [4]:
trainer = ModelTrainer()
metrics = trainer.train_and_evaluate(X_train, X_test, y_train, y_test)
trainer.save_plots(y_test)
trainer.save_artifacts(feature_names, fe.scaler)

comparison_df = trainer.generate_comparison_table()
comparison_df

### Visualizing Evaluation Metrics & ROC Curves

In [5]:
from IPython.display import Image, display
display(Image(filename='reports/model_metrics_comparison.png'))
display(Image(filename='reports/confusion_matrices_comparison.png'))
display(Image(filename='reports/roc_curves_comparison.png'))

## 4. SHAP Model Explainability
Using SHAP `TreeExplainer` on the gradient boosted model to uncover:
- Global feature impact (Beeswarm)
- Feature importance rankings
- Individual customer risk waterfalls

In [6]:
target_model = trainer.models.get('XGBoost', trainer.best_model)
explainer = ShapExplainer(target_model, feature_names)
explanation, X_sample = explainer.explain(X_test)
top_drivers = explainer.generate_plots(explanation, X_sample)

display(Image(filename='reports/shap_summary_beeswarm.png'))
display(Image(filename='reports/shap_feature_importance_bar.png'))

### Local Customer Risk Explanations

In [7]:
display(Image(filename='reports/shap_waterfall_high_risk.png'))
display(Image(filename='reports/shap_waterfall_low_risk.png'))

## Summary & Next Steps (Week 3 Preparation)
- **Key Finding**: Month-to-month contracts, fiber optic without tech support, bill shock, and electronic check payments are the strongest positive drivers of customer churn.
- **Champion Model**: Random Forest & XGBoost both demonstrated strong retention recall (>77%) and ROC-AUC (~0.85).
- **Week 3 Integration**: The saved model artifacts (`models/best_churn_model.pkl`) will be integrated into the FastAPI service for real-time customer lifetime value and churn risk scoring.